# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/swiftcoder0/flyrank_intership_assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Signal 1 — Staleness

**Verdict: OPPOSITE**

I expected pages that had not been updated for 180 days or more to have a higher declining rate. The data shows the opposite: stale pages had a 47.1% declining rate (n=174), while non-stale pages had a 54.2% declining rate (n=29,826).

This means staleness alone does not support prioritizing pages for refresh in this dataset. I will not rely on this signal as a strong positive reason in my baseline rule.

In [5]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/swiftcoder0/flyrank_intership_assignment.git"
REPO_DIR = "flyrank_intership_assignment"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print(
    "Dataset found:",
    os.path.exists("data/raw/content_refresh_anonymized.csv")
)

Working directory: /content/flyrank_intership_assignment/flyrank_intership_assignment/flyrank_intership_assignment
Dataset found: True


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset:", df.shape)

Dataset: (30000, 44)


In [7]:
# Signal 1: staleness
df["stale"] = df["days_since_last_update"] >= 180

stale_table = (
    df.groupby("stale")
      .agg(
          n=("stale", "size"),
          declining_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

print(stale_table)

   stale      n  declining_rate
0  False  29826        0.542480
1   True    174        0.471264


In [8]:
# Signal 2: CTR by position tier

ctr_table = (
    df.groupby("position_tier")
      .agg(
          n=("position_tier", "size"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
      .sort_values("position_tier")
)

print(ctr_table.to_string(index=False))

position_tier     n  mean_ctr
         deep  1319  0.150212
       page_1 11814  0.652467
     page_3_5  7242  0.222484
     striking  7304  0.323239
        top_3  2321  1.483611


### Signal 2 — CTR vs position

**Verdict: CONFIRMED**

The data shows a strong directional relationship between position tier and mean CTR. The `top_3` group has the highest mean CTR at 1.484, while the `deep` group has a much lower mean CTR of 0.150. The other position tiers fall between these values.

This supports using CTR and position visibility as signals when prioritizing pages for review. However, this is an observed relationship, not proof that changing a page's position will cause its CTR to increase.

## 2. Build the ranked queue (writes the CSV)

### Baseline rule

I will prioritize pages with weaker search visibility for review. The score is based on position tier, with deeper positions receiving a higher priority score. I will not use staleness as a positive signal because the signal check was OPPOSITE.

**Reason code:** `LOW_VISIBILITY`

**Action:** `REVIEW`

This is a simple decision-support baseline, not a prediction of Google's behavior. The score is intended to rank pages for human review.

In [13]:
# Build the baseline ranked queue.
# Rule: prioritize pages with weaker search visibility.

position_score = {
    "top_3": 0,
    "page_1": 25,
    "striking": 50,
    "page_3_5": 75,
    "deep": 100
}

df["baseline_score"] = df["position_tier"].map(position_score).fillna(50)

# One reason code and one action for every ranked page
df["reason_code"] = "LOW_VISIBILITY"
df["action"] = "REVIEW"

queue = (
    df.sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

queue_output = queue[
    [
        "rank",
        "content_id",
        "position_tier",
        "impressions_90d",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

queue_output.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows ranked: {len(queue_output):,}")
print("\nTop 20:")
print(queue_output.head(20).to_string(index=False))

print(
    queue_output[queue_output["position_tier"] == "deep"]
    [["rank", "content_id", "position_tier", "baseline_score"]]
    .head(25)
    .to_string(index=False)
)

print("\nUnique deep scores:")
print(queue_output.loc[
    queue_output["position_tier"] == "deep",
    "baseline_score"
].unique())

Saved: work/outputs/baseline_action_score.csv
Rows ranked: 30,000

Top 20:
 rank           content_id position_tier  impressions_90d  ctr  baseline_score    reason_code action
    1 content_a023517539fe          deep           214047 0.01             100 LOW_VISIBILITY REVIEW
    2 content_109f8f7c9d39          deep            90476 0.01             100 LOW_VISIBILITY REVIEW
    3 content_bcf8e8e2280d          deep            32820 0.01             100 LOW_VISIBILITY REVIEW
    4 content_fb66dd8f4629          deep            32518 0.09             100 LOW_VISIBILITY REVIEW
    5 content_62abc4bd66be          deep            31364 0.09             100 LOW_VISIBILITY REVIEW
    6 content_df71843dcd17          deep            27334 0.00             100 LOW_VISIBILITY REVIEW
    7 content_d49c7fc84373          deep            19859 0.02             100 LOW_VISIBILITY REVIEW
    8 content_58cffeeaa478          deep            19579 0.05             100 LOW_VISIBILITY REVIEW
    9 content_9e

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# Top-20 skeptical review based on the actual ranked values

top20_review = queue_output.head(20).copy()

def confidence_note(row):
    if row["ctr"] <= 0.02 and row["impressions_90d"] >= 10000:
        return "Higher-priority signal: deep position with substantial impressions and very low CTR."
    elif row["impressions_90d"] >= 10000:
        return "Moderate signal: deep position with substantial impressions, but CTR is not extremely low."
    else:
        return "Weaker signal: deep position, but lower impression volume makes the priority less certain."

def wrong_reason(row):
    if row["impressions_90d"] < 10000:
        return "Low impression volume may make the opportunity less important than the ranking suggests."
    elif row["ctr"] > 0.05:
        return "CTR is not extremely low, so low visibility may not indicate a clear refresh opportunity."
    else:
        return "The page may be intentionally targeting a low-visibility query, or other factors may explain its position."

top20_review["confidence_note"] = top20_review.apply(confidence_note, axis=1)
top20_review["what_would_make_it_wrong"] = top20_review.apply(wrong_reason, axis=1)

top20_review = top20_review[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print(top20_review.to_string(index=False))

 rank           content_id action    reason_code                                                                            confidence_note                                                                                   what_would_make_it_wrong
    1 content_a023517539fe REVIEW LOW_VISIBILITY       Higher-priority signal: deep position with substantial impressions and very low CTR. The page may be intentionally targeting a low-visibility query, or other factors may explain its position.
    2 content_109f8f7c9d39 REVIEW LOW_VISIBILITY       Higher-priority signal: deep position with substantial impressions and very low CTR. The page may be intentionally targeting a low-visibility query, or other factors may explain its position.
    3 content_bcf8e8e2280d REVIEW LOW_VISIBILITY       Higher-priority signal: deep position with substantial impressions and very low CTR. The page may be intentionally targeting a low-visibility query, or other factors may explain its position.
    4 conten

### Weak picks and leakage check

The weaker picks are the lower-impression pages near the bottom of the Top-20. They still have the `deep` position signal, but their lower impression volume makes the priority less certain.

The baseline uses `position_tier` and `impressions_90d` only. I deliberately exclude `trend_direction` and `trend_pct` because they describe the outcome being investigated and could leak future information into the rule.

This baseline is therefore decision-support rather than a claim that these pages will improve if changed.

In [16]:
# Section 4: identify weak picks and check for leakage

# Weak picks: lower-impression pages in the Top 20
weak_picks = top20_review[
    top20_review["rank"] >= 16
]

print("Weak picks:")
print(
    weak_picks[
        ["rank", "content_id", "confidence_note", "what_would_make_it_wrong"]
    ].to_string(index=False)
)

# Leakage check: confirm the baseline does not use outcome-derived fields.
baseline_features = [
    "position_tier",
    "impressions_90d"
]

leakage_fields = [
    "trend_direction",
    "trend_pct"
]

print("\nBaseline features used:")
print(baseline_features)

print("\nOutcome-derived fields checked:")
print(leakage_fields)

print("\nLeakage check:")
for field in leakage_fields:
    print(f"{field}: {'NOT USED' if field not in baseline_features else 'LEAKED'}")

Weak picks:
 rank           content_id                                                                            confidence_note                                                                 what_would_make_it_wrong
   16 content_e09b5602ba42 Weaker signal: deep position, but lower impression volume makes the priority less certain. Low impression volume may make the opportunity less important than the ranking suggests.
   17 content_011ba6714bb6 Weaker signal: deep position, but lower impression volume makes the priority less certain. Low impression volume may make the opportunity less important than the ranking suggests.
   18 content_ec5e5f49929b Weaker signal: deep position, but lower impression volume makes the priority less certain. Low impression volume may make the opportunity less important than the ranking suggests.
   19 content_3250919d61a0 Weaker signal: deep position, but lower impression volume makes the priority less certain. Low impression volume may make the opportu

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.